In [ ]:
# # Environment Testing :
# from langchain_openai import ChatOpenAI
# import os 
# from dotenv import load_dotenv

# load_dotenv()

# GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

# llm = ChatOpenAI(
#     model="gpt-4.1-mini",
#     base_url="https://models.github.ai/inference",
#     temperature=0.1,
#     max_tokens=500,
#     api_key=GITHUB_TOKEN
# )

# response = llm.invoke("What is the capital of India?")
# print(response.content)

### Read PDF Files:

In [ ]:
# Load Libraries:
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import os 

# Read All Pdf's :
def process_pdf(pdf_direcory):
    
    all_documents = []
    pdf_dir = Path(pdf_direcory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} pdf process")
    
    for p_file in pdf_files:
        print(f"\nProcess : {p_file}")

        try:
            loader = PyMuPDFLoader(str(p_file))
            documents = loader.load()

            # Add Source Info for metadata:
            for doc in documents:
                doc.metadata['source_file'] = p_file.name
                doc.metadata['file_type'] = "pdf"

            all_documents.extend(documents)
            print(f"💯Loded {len(documents)} Pages")
        
        except Exception as e:
            print(f"✖️ Error {e}")
        
    print(f"Total Documents Loded: {len(all_documents)}")
    return all_documents



# all_pdf_documents = process_pdf(r"D:\Code\AI-ML\Conversational_RAG\Data")
all_pdf_documents = process_pdf(r"D:\Code\Python\Intership_Learning\AI_Ml_Projects\04_Conversational_RAG\Data")
# all_pdf_documents = process_pdf(r"D:\Code\AI-ML\Conversational_RAG\Vectore_DB")

### Chunking Documnets:

In [ ]:
# Text Splitting get into chunks
def split_documnets(documnets, chunk_size=1000, overlap = 200):
    """Split Documents into Chunks."""
    text_chunk = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap = overlap,
        length_function = len,
        separators= ["\n\n","\n"," ", ""] #\n\n is a Peragraph Seprator
    )

    split_doc = text_chunk.split_documents(documnets)
    print(f"Split {len(documnets)} Documnets into {len(split_doc)} chunks.")

    if split_doc:
        print(f"\nExample Chunk:\nContent : \n{split_doc[0].page_content[:200]}...\nMetaData: {split_doc[0].metadata}")
    
    return split_doc

chunks = split_documnets(all_pdf_documents)


### vectore Store:

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import SentenceTransformerEmbeddings
import os

def vector_store(chunks, save_path="faiss_index"):
    
    print("\n🔄 Creating Embeddings (FAISS)...")

    embedding_model = SentenceTransformerEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    vector_db = FAISS.from_documents(
        documents=chunks,
        embedding=embedding_model
    )

    # Ensure directory exists
    os.makedirs(save_path, exist_ok=True)

    # Save locally
    vector_db.save_local(save_path)

    print(f"✅ FAISS Vector DB Created with {len(chunks)} chunks")
    print(f"📁 Saved at: {save_path}")

    return vector_db

# vectore_db = vector_store(chunks,"D:\Code\AI-ML\Conversational_RAG\Vectore_DB")
vectore_db = vector_store(chunks,r"D:\Code\Python\Intership_Learning\AI_Ml_Projects\04_Conversational_RAG\Vectore_DB")

### Chat with LLM:

In [ ]:
# from langchain_community.chat_message_histories import ChatMessageHistory
# from langchain_openai import ChatOpenAI
# from langchain_community.vectorstores import FAISS
# from langchain_community.embeddings import SentenceTransformerEmbeddings
# from dotenv import load_dotenv
# import json
# import os

# load_dotenv()

# NVDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

# # ✅ Shared embedding model (same as PDF)
# embedding_model = SentenceTransformerEmbeddings(
#     model_name="all-MiniLM-L6-v2"
# )

# # ✅ User knowledge vector DB (EMPTY INIT)
# user_knowledge_db = FAISS.from_texts(
#     texts=[""],   # dummy init (FAISS needs at least 1)
#     embedding=embedding_model
# )


# # 🔥 Query Rewriting (keep this)
# def rewrite_query(llm, chat_history, query):
#     prompt = f"""
#         Rewrite the question to be standalone.

#         CHAT HISTORY:
#         {chat_history}

#         QUESTION:
#         {query}

#         REWRITTEN QUESTION:
#         """
#     response = llm.invoke(prompt)
#     return response.content.strip()

# LOG_FILE = r"D:\Code\Python\Intership_Learning\AI_Ml_Projects\04_Conversational_RAG\conversations.txt"

# # ensure folder exists
# os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)

# def save_chat_json(user, bot):
#     data = {
#         "user": user,
#         "bot": bot
#     }

#     with open(LOG_FILE, "a", encoding="utf-8") as f:
#         f.write(json.dumps(data) + "\n")

# def start_chat(vector_db):
#     """Conversational RAG with dynamic user learning"""

#     print("\n🤖 Chatbot Started (type 'exit' to stop)\n")

#     # ✅ LLM
#     # llm = ChatOpenAI(
#     #     model="gpt-4.1-mini",
#     #     base_url="https://models.github.ai/inference",
#     #     temperature=0.1,
#     #     max_tokens=500,
#     #     api_key=os.getenv("GITHUB_TOKEN")
#     # )
#     llm = ChatOpenAI(
#         model="mistralai/mistral-7b-instruct-v0.3",
#         base_url="https://integrate.api.nvidia.com/v1",
#         temperature=0.0,
#         max_tokens=1000,
#         api_key=os.getenv("NVIDIA_API_KEY")
#     )

#     # ✅ Memory (for rewriting only)
#     memory = ChatMessageHistory()

#     # ✅ PDF retriever
#     retriever = vector_db.as_retriever(search_kwargs={"k": 3})

#     while True:
#         query = input("User: ")

#         if query.lower() == "exit":
#             print("\n👋 Exiting chatbot...")
#             break

#         # 🧠 Build chat history (limited)
#         chat_history = "\n".join([
#             f"User: {msg.content}" if msg.type == "human" else f"Bot: {msg.content}"
#             for msg in memory.messages[-4:]
#         ])

#         # 🔥 Rewrite query
#         if chat_history:
#             rewritten_query = rewrite_query(llm, chat_history, query)
#         else:
#             rewritten_query = query

#         # 🔍 Retrieve from PDF
#         doc_results = retriever.invoke(rewritten_query)

#         # 🔍 Retrieve from USER knowledge
#         try:
#             user_results = user_knowledge_db.similarity_search(rewritten_query, k=2)
#         except:
#             user_results = []

#         # 🔥 Combine both
#         docs = doc_results + user_results

#         if not docs:
#             answer = "I don't know based on the provided documents."
#         else:
#             # 📄 Build context
#             context = "\n\n".join([doc.page_content for doc in docs if doc.page_content.strip()])

#             # ✅ Final prompt (both sources included)
#             final_prompt = f"""
#                 Answer the question using the context.

#                 Rules:
#                 - Be concise but complete
#                 - Avoid repetition
#                 - Use bullet points if helpful
#                 - If not found, say:
#                 "I don't know based on the provided documents."

#                 CONTEXT:
#                 {context}

#                 QUESTION:
#                 {query}

#                 ANSWER:
#                 """
#             response = llm.invoke(final_prompt)
#             answer = response.content.strip()

#         # 💾 Save conversation memory
#         memory.add_user_message(query)
#         memory.add_ai_message(answer)

#         # 🧠 Store user input as knowledge (AFTER answering)
#         if query.strip():
#             user_knowledge_db.add_texts([query])

#         # ✅ Clean output
#         # print(f"\n👤User: {query}")
#         # print(f"🤖Bot: {answer}\n")
#         memory.add_user_message(query)
#         memory.add_ai_message(answer)

#         # ✅ SAVE EVERY RESPONSE (no truncation ever)
#         save_chat_json(query, answer)

#         print(f"\n👤User: {query}")
#         print(f"🤖Bot: {answer}\n")


In [ ]:
# from langchain_community.chat_message_histories import ChatMessageHistory
# from langchain_openai import ChatOpenAI
# from dotenv import load_dotenv
# import json
# import os

# load_dotenv()

# # ==============================
# # 🔒 Grounding Check Function
# # ==============================
# def is_grounded(answer, context):
#     answer_words = set(answer.lower().split())
#     context_words = set(context.lower().split())

#     overlap = answer_words.intersection(context_words)

#     return len(overlap) / max(len(answer_words), 1) > 0.4


# # ==============================
# # 💾 Logging Setup
# # ==============================
# LOG_FILE = r"D:\Code\Python\Intership_Learning\AI_Ml_Projects\04_Conversational_RAG\conversations.txt"
# os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)

# def save_chat_json(user, bot):
#     data = {"user": user, "bot": bot}
#     with open(LOG_FILE, "a", encoding="utf-8") as f:
#         f.write(json.dumps(data) + "\n")


# # ==============================
# # 🔥 Query Rewriting
# # ==============================
# def rewrite_query(llm, chat_history, query):
#     prompt = f"""
# Rewrite the question to be standalone.

# CHAT HISTORY:
# {chat_history}

# QUESTION:
# {query}

# REWRITTEN QUESTION:
# """
#     return llm.invoke(prompt).content.strip()


# # ==============================
# # 🚀 MAIN CHAT FUNCTION (FIXED)
# # ==============================
# def start_chat(vector_db):

#     print("\n🤖 Chatbot Started (type 'exit' to stop)\n")

#     llm = ChatOpenAI(
#         model="mistralai/mistral-7b-instruct-v0.3",
#         base_url="https://integrate.api.nvidia.com/v1",
#         temperature=0.0,
#         max_tokens=800,   # balanced (no truncation + cost control)
#         api_key=os.getenv("NVIDIA_API_KEY")
#     )

#     memory = ChatMessageHistory()

#     while True:
#         query = input("User: ")

#         if query.lower() == "exit":
#             print("\n👋 Exiting chatbot...")
#             break

#         # ==============================
#         # 🧠 Chat History (limited)
#         # ==============================
#         chat_history = "\n".join([
#             f"User: {msg.content}" if msg.type == "human" else f"Bot: {msg.content}"
#             for msg in memory.messages[-4:]
#         ])

#         # ==============================
#         # 🔥 Query Rewrite
#         # ==============================
#         rewritten_query = rewrite_query(llm, chat_history, query) if chat_history else query

#         # ==============================
#         # 🔍 Retrieval with Scores
#         # ==============================
#         docs_with_scores = vector_db.similarity_search_with_score(rewritten_query, k=5)

#         # Filter relevant docs (lower score = better)
#         filtered_docs = [doc for doc, score in docs_with_scores if score < 1.2]

#         # fallback if nothing passes
#         if not filtered_docs:
#             filtered_docs = [doc for doc, _ in docs_with_scores[:2]]

#         print(f"Retrieved Docs: {len(filtered_docs)}")

#         # ==============================
#         # 📄 Build Context (top 3 only)
#         # ==============================
#         top_docs = filtered_docs[:3]

#         context = "\n\n".join([
#             doc.page_content.strip()
#             for doc in top_docs
#             if doc.page_content.strip()
#         ])

#         # ==============================
#         # 🤖 LLM Answer (STRICT PROMPT)
#         # ==============================
#         final_prompt = f"""
#             Answer the question ONLY using the given context.

#             Rules:
#             - Do NOT use outside knowledge
#             - Do NOT guess or assume
#             - If answer is not clearly in context, say:
#             "I don't know based on the provided documents."
#             - Be concise and complete

#             CONTEXT:
#             {context}

#             QUESTION:
#             {query}

#             ANSWER:
#             """

#         response = llm.invoke(final_prompt)
#         answer = response.content.strip()

#         # ==============================
#         # 🔒 Grounding Check
#         # ==============================
#         if not is_grounded(answer, context):
#             answer = "I don't know based on the provided documents."

#         # ==============================
#         # 💾 Memory Save (FIXED: no duplicate)
#         # ==============================
#         memory.add_user_message(query)
#         memory.add_ai_message(answer)

#         # ==============================
#         # 💾 File Logging
#         # ==============================
#         save_chat_json(query, answer)

#         # ==============================
#         # 🖥️ Output
#         # ==============================
#         print(f"\n👤User: {query}")
#         print(f"🤖Bot: {answer}\n")

In [17]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from sentence_transformers import CrossEncoder
import json
import os

load_dotenv()

# ==============================
# 🔥 RERANKER (NEW)
# ==============================
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# ==============================
# 🔒 Grounding Check (RELAXED)
# ==============================
def is_grounded(answer, context):
    answer_words = set(answer.lower().split())
    context_words = set(context.lower().split())

    overlap = answer_words.intersection(context_words)

    return len(overlap) / max(len(answer_words), 1) > 0.3   # 🔥 changed 0.4 → 0.3


# ==============================
# 💾 Logging
# ==============================
LOG_FILE = r"D:\Code\Python\Intership_Learning\AI_Ml_Projects\04_Conversational_RAG\conversations.txt"
os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)

def save_chat_json(user, bot):
    data = {"user": user, "bot": bot}
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(data) + "\n")


# ==============================
# 🔥 Query Rewriting
# ==============================
def rewrite_query(llm, chat_history, query):
    prompt = f"""
Rewrite the question to be standalone.

CHAT HISTORY:
{chat_history}

QUESTION:
{query}

REWRITTEN QUESTION:
"""
    return llm.invoke(prompt).content.strip()


# ==============================
# 🚀 MAIN CHAT FUNCTION (FINAL)
# ==============================
def start_chat(vector_db):

    print("\n🤖 Chatbot Started (type 'exit' to stop)\n")

    llm = ChatOpenAI(
        model="mistralai/mistral-7b-instruct-v0.3",
        base_url="https://integrate.api.nvidia.com/v1",
        temperature=0.0,
        max_tokens=800,
        api_key=os.getenv("NVIDIA_API_KEY")
    )

    memory = ChatMessageHistory()

    while True:
        query = input("User: ")

        if query.lower() == "exit":
            print("\n👋 Exiting chatbot...")
            break

        # ==============================
        # 🧠 Chat History
        # ==============================
        chat_history = "\n".join([
            f"User: {msg.content}" if msg.type == "human" else f"Bot: {msg.content}"
            for msg in memory.messages[-4:]
        ])

        # ==============================
        # 🔥 Query Rewrite
        # ==============================
        rewritten_query = rewrite_query(llm, chat_history, query) if chat_history else query

        # ==============================
        # 🔍 Retrieval (FAISS)
        # ==============================
        docs_with_scores = vector_db.similarity_search_with_score(rewritten_query, k=5)

        docs = [doc for doc, _ in docs_with_scores]

        # ==============================
        # 🔥 RERANKING (NEW CORE IMPROVEMENT)
        # ==============================
        pairs = [(rewritten_query, doc.page_content) for doc in docs]
        scores = reranker.predict(pairs)

        reranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

        # Take top 3 best docs
        top_docs = [doc for doc, _ in reranked[:3]]

        print(f"Retrieved Docs after rerank: {len(top_docs)}")

        # ==============================
        # 📄 Build Context
        # ==============================
        context = "\n\n".join([
            doc.page_content.strip()
            for doc in top_docs
            if doc.page_content.strip()
        ])

        # ==============================
        # 🤖 LLM Answer (IMPROVED PROMPT)
        # ==============================
        final_prompt = f"""
Answer the question using ONLY the given context.

Rules:
- Do NOT use outside knowledge
- If partially available, answer what is present
- Do NOT guess missing information
- If answer not found, say:
"I don't know based on the provided documents."
- Be concise and complete

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""

        response = llm.invoke(final_prompt)
        answer = response.content.strip()

        # ==============================
        # 🔒 Grounding Check
        # ==============================
        if not is_grounded(answer, context):
            answer = "I don't know based on the provided documents."

        # ==============================
        # 💾 Memory
        # ==============================
        memory.add_user_message(query)
        memory.add_ai_message(answer)

        # ==============================
        # 💾 Logging
        # ==============================
        save_chat_json(query, answer)

        # ==============================
        # 🖥️ Output
        # ==============================
        print(f"\n👤User: {query}")
        print(f"🤖Bot: {answer}\n")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
# ✅ RUN
chat = start_chat(vectore_db)


🤖 Chatbot Started (type 'exit' to stop)

Retrieved Docs after rerank: 3

👤User: What is an outlier in data analysis?
🤖Bot: An outlier in data analysis is an observation that is rare, distinct, or does not fit in some way compared to other observations in a dataset. It can be an extreme value or unusual data point that significantly differs from other observations.

Retrieved Docs after rerank: 3

👤User: Why are outliers important in data analysis?
🤖Bot: Outliers are important in data analysis because they can bias analysis and model training, influence statistical measures, reveal anomalies or errors, and provide valuable insights such as detecting fraudulent transactions or identifying unusual customer behavior.

Retrieved Docs after rerank: 3

👤User: What are common methods used to detect outliers?
🤖Bot: Common methods used to detect outliers include DBSCAN (Density-Based Spatial Clustering of Applications with Noise) and visual methods such as Box Plots.

Retrieved Docs after reran